# 04. Resource Allocation & Policy Optimization

**Theme C: Data-Driven Resource Allocation of Insecticide-Treated Nets (ITNs)**  
**Target:** Ghana National Malaria Elimination Programme (NMEP)  
**Objective:** Formulate an equitable, uncertainty-aware ITN allocation policy across 50 northern districts under a strict constraint of 50,000 nets.

---

### Decision Framing Summary:
1. **Target:** Populations in northern Ghana vulnerable to malaria transmission.
2. **Unit of Analysis:** District level ( = 50$ districts across Northern, Upper East, and Upper West regions).
3. **Objective Metric:** Minimize preventable malaria burden by prioritizing **unmet epidemiological need** (epidemic risk upper-bound $	imes$ unmet coverage gap).
4. **Operational Constraints:** Exactly 50,000 integer nets, zero negative allocations, single shipment delivery.
5. **Equity vs. Efficiency Trade-offs:** Avoid penalizing rural districts with poor health-seeking infrastructure (referral hospital bias).


In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src/ can be imported regardless of execution working directory
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src import io, models, viz

RANDOM_SEED = io.RANDOM_SEED
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore')
viz.set_theme()

print('Setup complete. Random seed set to', RANDOM_SEED)

## 1. Load District Surveillance & Coverage Data
We load the curated district dataset covering the 50 northern districts.


In [ ]:
district = io.load_district_cases()
print(f"Loaded {len(district)} districts across {district['region_name'].nunique()} regions.")
district[['district', 'region_name', 'mean_population', 'positive_cases', 'net_coverage_pct']].head(5)


## 2. Fit Negative Binomial Model with Population Offset
As established in Theme A (), malaria counts exhibit extreme over-dispersion (Variance/Mean $\approx 77,200$). 
A Poisson GLM is heavily rejected ($\chi^2/\text{df} \approx 54,100$). 

We fit a Negative Binomial regression with $\log(\text{population})$ offset to obtain transmission risk estimates and 95% confidence intervals.


In [ ]:
formula = "positive_cases ~ net_coverage_pct"
offset = np.log(district["mean_population"].clip(lower=1)).to_numpy()

# Fit working NB model
nb_fit = models.fit_negative_binomial_mle(formula, district, offset=offset)
print(f"Negative Binomial fit converged: {nb_fit.mle_retvals.get('converged')}")
print(f"Dispersion alpha (MLE): {float(nb_fit.params['alpha']):.4f}")


## 3. Allocation Policy: Naive vs. Equitable (Need-Weighted)

### The Naive Policy (Status Quo Baseline)
Allocates nets strictly in proportion to historical clinic-reported positive cases:
33019\text{Weight}_i^{\text{naive}} = \text{positive\_cases}_i33019
33019\text{Allocation}_i^{\text{naive}} = N \times \frac{\text{Weight}_i^{\text{naive}}}{\sum_j \text{Weight}_j^{\text{naive}}}33019

*Fatal Flaws of Naive Policy:*
1. **Referral Hub Distortion:** Regional tertiary hospitals (e.g. Bolgatanga Regional Hospital, Wa Regional Hospital) treat patients from surrounding rural districts, massively inflating the host district's reported case count.
2. **Ignoring Baseline Coverage:** If an urban referral hub already has 80% net coverage, dumping additional nets saturates already-protected households while rural catchment zones remain unprotected.
3. **Ignoring Epidemic Risk & Uncertainty:** Relies on a noisy point estimate rather than protecting against potential surge outbreaks.

---

### The Equitable Need-Weighted Policy
We formulate an allocation index that combines:
1. **Upper-Bound Predicted Risk:** $\hat{\mu}_{i,\text{upper}}$ (upper bound of the 95% prediction interval from the Negative Binomial model) to provide precautionary coverage against worst-case epidemic spikes.
2. **Unmet Coverage Gap:**  = 1 - \frac{\text{net\_coverage\_pct}_i}{100}$, directing resources where existing net coverage is lowest.
3. **Hamilton Integer Apportionment:** Exactly guarantees the 50,000 net budget ceiling with zero fractional nets.

33019\text{Weight}_i^{\text{equitable}} = \hat{\mu}_{i,\text{upper}} \times \left(1 - \frac{\text{coverage}_i}{100}\right)33019


In [ ]:
alloc_df = models.compute_allocation(district, nb_fit, total_nets=50000, offset=offset)

# Verify budget constraint
assert alloc_df['equitable_allocation'].sum() == 50000, "Equitable budget must sum to exactly 50,000"
assert alloc_df['naive_allocation'].sum() == 50000, "Naive budget must sum to exactly 50,000"

print(f"Budget verified: Exactly {alloc_df['equitable_allocation'].sum():,} nets allocated across {len(alloc_df)} districts.")


## 4. Empirical Evaluation: Top Gainers vs. Top Losers

Comparing the equitable allocation against the naive baseline reveals substantial policy reallocations ($\Delta = \text{Equitable} - \text{Naive}$).


In [ ]:
gainers = alloc_df.sort_values(by="delta", ascending=False).head(5)[
    ['district', 'region_name', 'mean_population', 'net_coverage_pct', 'naive_allocation', 'equitable_allocation', 'delta']
]
losers = alloc_df.sort_values(by="delta", ascending=True).head(5)[
    ['district', 'region_name', 'mean_population', 'net_coverage_pct', 'naive_allocation', 'equitable_allocation', 'delta']
]

print("=== TOP 5 GAINERS (Districts receiving MORE nets under equitable policy) ===")
print(gainers.to_string(index=False))
print()
print("=== TOP 5 LOSERS (Districts receiving FEWER nets under equitable policy) ===")
print(losers.to_string(index=False))


## 5. Visualizing Policy Shifts
We visualize the reallocations using a comparative horizontal delta bar chart, exported for the Ministry presentation deck.


In [ ]:
fig, ax = viz.plot_allocation_comparison(alloc_df, n_top=5)
fig_path = viz.save_figure(fig, "c1_allocation_comparison.png")
print(f"Figure successfully saved to: {fig_path}")
plt.show()


## 6. Policy Recommendations & Oral Defense Takeaways

1. **Why Tamale (+1,692) and Sagnarigu (+1,012) Gain Nets:**
   - Tamale is Northern Ghana's most populous metropolitan area (~255,000 pop) and sits in a region with lower baseline coverage (~64%). Under naive case allocation, Tamale was severely under-allocated relative to its population at risk.
   - The equitable model recognizes both its large susceptible population and its unmet coverage gap.

2. **Why Wa (-1,516) and Bolgatanga (-665) Lose Nets:**
   - Wa and Bolgatanga host major regional referral hospitals. Patients travel from surrounding rural districts (e.g. Sissala, Nabdam, Bongo) to seek diagnosis and treatment.
   - Allocating nets purely on hospital test records erroneously credits all these infections to the urban hospital district!
   - Both Wa and Bolgatanga already possess higher baseline net ownership (~80%). Funneling 3,000+ nets there causes diminishing marginal health returns, while redistributing them to rural communities targets true community transmission.

3. **Defensibility Before the Panel:**
   - When asked: *"Why did you cut nets from Bolgatanga when it has the highest positive cases?"*
   - Defense: *"Bolgatanga's case counts reflect hospital catchment, not isolated district incidence. Furthermore, Upper East already has 80% net coverage. Our equitable model redirects nets to unprotected rural populations, maximizing community-wide transmission interruption."*
